# Metrics HasAns NoAns và answerability trên dev

Notebook này tính EM/F1 HasAns/NoAns cho các dự đoán đã lưu và tính accuracy/confusion matrix no-answer cho từng phương pháp. Không huấn luyện lại mô hình và không đọc test.json. Nhãn thật lấy từ dev.json; dự đoán ghép theo id. Kết quả được lưu trong results/dev_answerability_and_group_metrics.json.


## Bước 1 — Nạp dev và hàm metric dùng chung


In [1]:
import csv
import json
import sys
from pathlib import Path

# Xác định thư mục gốc dù kernel chạy từ notebooks hay thư mục dự án.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Cho phép import hàm đánh giá dùng trong pipeline fine-tune.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import score_prediction

# Mỗi dòng JSONL là một mẫu dev.
dev_path = project_root / "data" / "processed" / "dev.json"
dev_examples = []
with dev_path.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            dev_examples.append(json.loads(line))

# Tra cứu mẫu chuẩn theo id để ghép chính xác.
dev_by_id = {example["id"]: example for example in dev_examples}
dev_ids = set(dev_by_id)
if len(dev_by_id) != len(dev_examples):
    raise ValueError("dev.json có id bị lặp.")

print("Dev:", len(dev_examples))
print("HasAns:", sum(not row["is_impossible"] for row in dev_examples))
print("NoAns:", sum(bool(row["is_impossible"]) for row in dev_examples))


Dev: 3814
HasAns: 2653
NoAns: 1161


## Bước 2 — Đọc predictions đã lưu
Jaccard nằm trong CSV; các phương pháp khác nằm trong JSON. Mỗi file phải có đúng một dự đoán cho từng id trong dev. Nếu sau này có predictions PhoBERT, notebook tự thêm các seed đó.


In [10]:
def read_json_predictions(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

def read_csv_predictions(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))

prediction_files = {
    "Jaccard": project_root / "results" / "baseline_predictions.csv",
    "TF-IDF + Logistic Regression": project_root / "results" / "baseline_tfidf_logreg_predictions.json",
    "Zero-shot": project_root / "results" / "zeroshot_predictions.json",
}

# Sau khi train đủ ba seed, điền tên run-... tương ứng từng kiến trúc.
# Giá trị None đọc thư mục kết quả lịch sử của mô hình đó.
versioned_runs = {"xlm-roberta": "run-fe5f5f5b27578bd4",
    "mdeberta": "run-a28388a1c7754133"}
finetune_root = project_root / "results" / "finetune"

for model_name, display_name in [("xlm-roberta", "XLM-R"), ("mdeberta", "mDeBERTa")]:
    experiment_id = versioned_runs[model_name]
    if experiment_id is None:
        model_results_dir = finetune_root / model_name
        print(f"{display_name}: đọc thư mục kết quả lịch sử.")
    else:
        # Đảm bảo dev của lượt mới đúng với dev được dùng để chấm ở notebook này.
        from hashlib import sha256

        experiment_dir = finetune_root / "runs" / experiment_id
        with (experiment_dir / "run_manifest.json").open(encoding="utf-8") as file:
            manifest = json.load(file)
        assert manifest["experiment_id"] == experiment_id
        assert model_name in manifest["sources"]
        dev_hash = sha256(dev_path.read_bytes()).hexdigest()
        if manifest["data"]["splits"]["dev"]["sha256"] != dev_hash:
            raise ValueError(f"{display_name}: SHA-256 của dev không khớp {experiment_id}.")
        model_results_dir = experiment_dir / model_name
        print(f"{display_name}: đọc {experiment_id}.")

    for seed in (42, 123, 1024):
        prediction_files[f"{display_name} seed {seed}"] = (
            model_results_dir / f"seed-{seed}" / "predictions.json"
        )

# Nếu đã có dự đoán PhoBERT thì tự thêm vào danh sách.
phobert_dir = project_root / "results" / "finetune" / "phobert"
for path in sorted(phobert_dir.glob("seed-*/predictions.json")):
    seed = path.parent.name.removeprefix("seed-")
    prediction_files[f"PhoBERT seed {seed}"] = path

# Đọc từng file; nếu file bắt buộc thiếu, thông báo rõ đường dẫn.
predictions_by_method = {}
for method, path in prediction_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file dự đoán: {path}")

    if path.suffix.lower() == ".csv":
        rows = read_csv_predictions(path)
    else:
        rows = read_json_predictions(path)

    ids = [row["id"] for row in rows]
    id_set = set(ids)
    if len(id_set) != len(ids):
        raise ValueError(f"{method} có id dự đoán bị lặp.")

    if id_set != dev_ids:
        missing = dev_ids - id_set
        extra = id_set - dev_ids
        raise ValueError(
            f"{method}: id không khớp dev; thiếu {len(missing)}, "
            f"thừa {len(extra)}."
        )

    predictions_by_method[method] = rows
    print(f"{method}: {len(rows)} dự đoán; id khớp dev=True")


XLM-R: đọc run-fe5f5f5b27578bd4.
mDeBERTa: đọc run-a28388a1c7754133.
Jaccard: 3814 dự đoán; id khớp dev=True
TF-IDF + Logistic Regression: 3814 dự đoán; id khớp dev=True
Zero-shot: 3814 dự đoán; id khớp dev=True
XLM-R seed 42: 3814 dự đoán; id khớp dev=True
XLM-R seed 123: 3814 dự đoán; id khớp dev=True
XLM-R seed 1024: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 42: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 123: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 1024: 3814 dự đoán; id khớp dev=True
PhoBERT seed 1024: 3814 dự đoán; id khớp dev=True
PhoBERT seed 123: 3814 dự đoán; id khớp dev=True
PhoBERT seed 42: 3814 dự đoán; id khớp dev=True


## Bước 3 — Tính HasAns/NoAns EM/F1 và confusion matrix
HasAns nghĩa là nhãn thật is_impossible=False; NoAns nghĩa là True. Dự đoán NoAns được xác định bằng prediction_text == chuỗi rỗng. Confusion matrix có hàng là nhãn thật, cột là nhãn dự đoán, theo thứ tự HasAns rồi NoAns.


In [11]:
results = {}

for method, rows in predictions_by_method.items():
    # Dùng id thay vì trông vào thứ tự dòng của file.
    prediction_by_id = {row["id"]: row for row in rows}

    scores = {
        "HasAns": {"em": [], "f1": []},
        "NoAns": {"em": [], "f1": []},
    }

    # Bốn ô ma trận khi coi NoAns là lớp dương.
    true_has_pred_has = 0
    true_has_pred_no = 0
    true_no_pred_has = 0
    true_no_pred_no = 0

    for example_id, gold in dev_by_id.items():
        prediction_text = prediction_by_id[example_id]["prediction_text"]
        is_noans_true = bool(gold["is_impossible"])
        is_noans_pred = prediction_text == ""

        group = "NoAns" if is_noans_true else "HasAns"
        em, f1 = score_prediction(
            prediction_text,
            gold["answers"].get("text", []),
        )
        scores[group]["em"].append(em)
        scores[group]["f1"].append(f1)

        # Cập nhật confusion matrix: hàng nhãn thật, cột dự đoán.
        if not is_noans_true and not is_noans_pred:
            true_has_pred_has += 1
        elif not is_noans_true and is_noans_pred:
            true_has_pred_no += 1
        elif is_noans_true and not is_noans_pred:
            true_no_pred_has += 1
        else:
            true_no_pred_no += 1

    # Tính điểm trung bình theo nhóm và đổi sang phần trăm.
    group_metrics = {}
    for group, values in scores.items():
        count = len(values["em"])
        group_metrics[group] = {
            "count": count,
            "EM_percent": 100 * sum(values["em"]) / count,
            "F1_percent": 100 * sum(values["f1"]) / count,
        }

    # Hàng: nhãn thật HasAns/NoAns; cột: dự đoán HasAns/NoAns.
    matrix = [
        [true_has_pred_has, true_has_pred_no],
        [true_no_pred_has, true_no_pred_no],
    ]
    total = len(dev_examples)
    accuracy = 100 * (true_has_pred_has + true_no_pred_no) / total
    noans_total = true_no_pred_has + true_no_pred_no
    noans_recall = 100 * true_no_pred_no / noans_total if noans_total else 0.0

    results[method] = {
        "groups": group_metrics,
        "answerability_accuracy_percent": accuracy,
        "no_answer_recall_percent": noans_recall,
        "confusion_matrix_labels": ["HasAns", "NoAns"],
        "confusion_matrix": matrix,
    }

    print()
    print(method)
    for group in ["HasAns", "NoAns"]:
        metric = group_metrics[group]
        print(
            f"{group}: {metric['count']} mẫu | "
            f"EM={metric['EM_percent']:.2f}% | "
            f"F1={metric['F1_percent']:.2f}%"
        )
    print(f"Accuracy answerability: {accuracy:.2f}%")
    print(f"Recall NoAns: {noans_recall:.2f}%")
    print("Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:")
    print(matrix)



Jaccard
HasAns: 2653 mẫu | EM=1.62% | F1=32.32%
NoAns: 1161 mẫu | EM=0.00% | F1=0.00%
Accuracy answerability: 69.56%
Recall NoAns: 0.00%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2653, 0], [1161, 0]]

TF-IDF + Logistic Regression
HasAns: 2653 mẫu | EM=1.06% | F1=23.03%
NoAns: 1161 mẫu | EM=23.43% | F1=23.43%
Accuracy answerability: 58.36%
Recall NoAns: 23.43%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[1954, 699], [889, 272]]

Zero-shot
HasAns: 2653 mẫu | EM=39.65% | F1=58.93%
NoAns: 1161 mẫu | EM=45.65% | F1=45.65%
Accuracy answerability: 68.59%
Recall NoAns: 45.65%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2086, 567], [631, 530]]

XLM-R seed 42
HasAns: 2653 mẫu | EM=44.40% | F1=58.51%
NoAns: 1161 mẫu | EM=34.71% | F1=34.71%
Accuracy answerability: 63.37%
Recall NoAns: 34.71%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2014, 639], [758, 403]]

XLM-R seed 123
HasAns: 2653 mẫu | EM=42.18% | F1=57.08%
NoAns: 116

## Bước 4 — Lưu kết quả
Accuracy là tỷ lệ đúng trên cả hai lớp. Recall NoAns cho biết tỷ lệ câu no-answer thật sự được nhận ra. Ma trận được in theo hàng thật [HasAns, NoAns] và cột dự đoán [HasAns, NoAns]; có thể đọc lần lượt là TN, FP, FN, TP khi coi NoAns là lớp dương.


In [17]:
output = {
    "split": "dev",
    "sample_count": len(dev_examples),
    "no_answer_rule": "prediction_text == empty string",
    "confusion_matrix_order": {
        "rows": ["true_HasAns", "true_NoAns"],
        "columns": ["predicted_HasAns", "predicted_NoAns"],
    },
    "finetune_experiment_ids": versioned_runs,
    "methods": results,
}

output_path = (
    project_root / "results" / "dev_answerability_and_group_metrics.json"
)
with output_path.open("w", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2)

print("Đã lưu kết quả dev:", output_path)


Đã lưu kết quả dev: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\dev_answerability_and_group_metrics.json


In [14]:
from math import isclose  # So sánh số thập phân có sai số làm tròn.
from src.evaluation import score_prediction  # Hàm chấm EM/F1 đang có trong dự án.

# Mỗi dòng gồm tên ca, dự đoán, đáp án chuẩn, EM và F1 kỳ vọng.
cases = [
    ("Khớp hoàn toàn", "HÀ-NỘI!", ["Huế", "Hà Nội"], 1.0, 1.0),
    ("Khớp một phần", "Đại học", ["Đại học Quốc gia"], 0.0, 2 / 3),
    ("Không có đáp án, trả rỗng", "", [], 1.0, 1.0),
    ("Không có đáp án, vẫn trả lời", "Hà Nội", [], 0.0, 0.0),
]

for name, prediction, references, expected_em, expected_f1 in cases:
    em, f1 = score_prediction(prediction, references)
    print(f"{name} | EM={em:.2f} | F1={f1:.4f}")

    # Dừng tại ca đầu tiên có kết quả khác dự kiến.
    assert em == expected_em, f"{name}: EM khác dự kiến"
    assert isclose(f1, expected_f1), f"{name}: F1 khác dự kiến"

print("Đạt: cả 4 ca kiểm tra đều đúng.")

Khớp hoàn toàn | EM=1.00 | F1=1.0000
Khớp một phần | EM=0.00 | F1=0.6667
Không có đáp án, trả rỗng | EM=1.00 | F1=1.0000
Không có đáp án, vẫn trả lời | EM=0.00 | F1=0.0000
Đạt: cả 4 ca kiểm tra đều đúng.


In [15]:
examples = [
    (False, "Hà Nội"),  # Có đáp án, mô hình trả lời.
    (False, "Huế"),     # Có đáp án, mô hình trả lời.
    (False, ""),        # Có đáp án, mô hình bỏ sót.
    (True, "..."),      # Không có đáp án, mô hình vẫn trả chuỗi không rỗng.
    (True, ""),         # Không có đáp án, mô hình trả rỗng đúng.
]

# Hàng là nhãn thật. Cột là nhãn dự đoán.
# Chỉ số 0 là HasAns, chỉ số 1 là NoAns.
matrix = [[0, 0], [0, 0]]

for is_impossible, prediction_text in examples:
    predicted_noans = prediction_text == ""

    # int(False) = 0 và int(True) = 1, đúng với thứ tự của ma trận.
    row = int(is_impossible)
    column = int(predicted_noans)
    matrix[row][column] += 1

# Hai ô trên đường chéo là các quyết định đúng.
accuracy = (matrix[0][0] + matrix[1][1]) / len(examples)

# Hàng thứ hai gồm tất cả các câu thật sự không có đáp án.
noans_recall = matrix[1][1] / sum(matrix[1])

print("Ma trận:", matrix)
print(f"Accuracy toàn bộ: {accuracy:.2%}")
print(f"Recall NoAns: {noans_recall:.2%}")

assert matrix == [[2, 1], [1, 1]]
assert accuracy == 0.60
assert noans_recall == 0.50
print("Đạt kiểm tra.")

Ma trận: [[2, 1], [1, 1]]
Accuracy toàn bộ: 60.00%
Recall NoAns: 50.00%
Đạt kiểm tra.


In [16]:
# Kiểm tra tên hàm đang trỏ đúng về module dùng chung.
assert score_prediction.__module__ == "src.evaluation"

# Hàm chung đổi dấu gạch dưới thành khoảng trắng.
# Bộ chấm cũ của notebook 02 giữ dấu gạch dưới.
em, f1 = score_prediction("Hà_Nội", ["Hà Nội"])
assert (em, f1) == (1.0, 1.0)

print(f"T05: EM={em:.2f} | F1={f1:.2f}")

T05: EM=1.00 | F1=1.00


## Kết quả kiểm tra cần thấy
Dev phải có 3.814 mẫu, gồm 2.653 HasAns và 1.161 NoAns. Mỗi phương pháp phải báo 3.814 dự đoán và ID khớp dev=True. Kết quả gồm HasAns/NoAns EM/F1, answerability accuracy, Recall NoAns và confusion matrix. Đây là kết quả dev, không phải test.


## Bước 5 — Tạo bảng so sánh cuối trên dev

Overall EM/F1 được tính theo số mẫu của hai nhóm HasAns và NoAns. Sau đó, với mỗi kiến trúc fine-tune, tính trung bình và độ lệch chuẩn quần thể (chia cho số seed, ddof=0) từ ba seed có trọng số bằng nhau. Chỉ làm tròn khi hiển thị. Ma trận nhầm lẫn được lưu theo từng phương pháp và seed, với hàng là nhãn thật và cột là dự đoán theo thứ tự HasAns, NoAns.


In [18]:
import json
from statistics import mean, pstdev

import pandas as pd

from src.utils import save_json

# Đọc báo cáo vừa được lưu ở Bước 4 để bảng có đúng mã lượt thí nghiệm.
input_path = project_root / "results" / "dev_answerability_and_group_metrics.json"
report = json.loads(input_path.read_text(encoding="utf-8"))
saved_runs = report.get("finetune_experiment_ids")
if report["split"] != "dev" or saved_runs != versioned_runs or not all(saved_runs.values()):
    raise ValueError("Chạy lại Bước 2–4 với đúng hai lượt XLM-R và mDeBERTa đã chốt.")

sample_count = report["sample_count"]
methods = report["methods"]
comparison_rows = []

for method, details in methods.items():
    has_ans = details["groups"]["HasAns"]
    no_ans = details["groups"]["NoAns"]
    if has_ans["count"] + no_ans["count"] != sample_count:
        raise ValueError(f"{method}: tổng hai nhóm không bằng số mẫu dev.")

    # Nhân điểm nhóm với số câu trong nhóm rồi chia cho tổng số câu.
    overall_em = (
        has_ans["EM_percent"] * has_ans["count"]
        + no_ans["EM_percent"] * no_ans["count"]
    ) / sample_count
    overall_f1 = (
        has_ans["F1_percent"] * has_ans["count"]
        + no_ans["F1_percent"] * no_ans["count"]
    ) / sample_count

    comparison_rows.append({
        "method": method,
        "overall_em_percent": overall_em,
        "overall_f1_percent": overall_f1,
        "has_ans_em_percent": has_ans["EM_percent"],
        "has_ans_f1_percent": has_ans["F1_percent"],
        "no_ans_em_percent": no_ans["EM_percent"],
        "no_ans_f1_percent": no_ans["F1_percent"],
        "answerability_accuracy_percent": details["answerability_accuracy_percent"],
        "no_answer_recall_percent": details["no_answer_recall_percent"],
        "confusion_matrix": details["confusion_matrix"],
    })

rows_by_method = {row["method"]: row for row in comparison_rows}
metric_names = [
    "overall_em_percent", "overall_f1_percent",
    "has_ans_em_percent", "has_ans_f1_percent",
    "no_ans_em_percent", "no_ans_f1_percent",
    "answerability_accuracy_percent", "no_answer_recall_percent",
]
table_labels = [
    "Overall EM (%)", "Overall F1 (%)",
    "HasAns EM (%)", "HasAns F1 (%)",
    "NoAns EM (%)", "NoAns F1 (%)",
    "Accuracy có/không đáp án (%)", "Recall NoAns (%)",
]

seed_list = [42, 123, 1024]
finetune_summary = {}
summary_rows = []

for architecture in ("XLM-R", "mDeBERTa", "PhoBERT"):
    seed_rows = []
    for seed in seed_list:
        method_name = f"{architecture} seed {seed}"
        if method_name not in rows_by_method:
            raise ValueError(f"Thiếu kết quả của {method_name}.")
        seed_rows.append(rows_by_method[method_name])

    # Tính từ điểm chưa làm tròn. pstdev là std quần thể (ddof=0).
    architecture_metrics = {}
    table_row = {"Kiến trúc": architecture}
    for metric, label in zip(metric_names, table_labels):
        values = [row[metric] for row in seed_rows]
        average = mean(values)
        std = pstdev(values)
        architecture_metrics[metric] = {"mean": average, "std": std}
        table_row[label] = f"{average:.2f} +- {std:.2f}"

    finetune_summary[architecture] = {
        "seeds": seed_list,
        "metrics": architecture_metrics,
    }
    summary_rows.append(table_row)

print(f"So sánh trên dev: {sample_count} mẫu mỗi bộ dự đoán.")
table = pd.DataFrame(comparison_rows)
display(table[["method", *metric_names]].rename(columns={
    "method": "Phương pháp / seed",
    **dict(zip(metric_names, table_labels)),
}).round(2))

print("Kết quả fine-tune qua 3 seed: trung bình +- độ lệch chuẩn (ddof=0).")
display(pd.DataFrame(summary_rows))

output = {
    "split": report["split"],
    "sample_count": sample_count,
    "finetune_experiment_ids": saved_runs,
    "std_ddof": 0,
    "no_answer_rule": report["no_answer_rule"],
    "confusion_matrix_order": report["confusion_matrix_order"],
    "methods": rows_by_method,
    "finetune_summary": finetune_summary,
}
comparison_path = project_root / "results" / "dev_model_comparison.json"
save_json(comparison_path, output)
print("Đã lưu bảng so sánh dev:", comparison_path)


So sánh trên dev: 3814 mẫu mỗi bộ dự đoán.


,Phương pháp / seed,Overall EM (%),Overall F1 (%),HasAns EM (%),HasAns F1 (%),NoAns EM (%),NoAns F1 (%),Accuracy có/không đáp án (%),Recall NoAns (%)
0,Jaccard,1.13,22.48,1.62,32.32,0.00,0.00,69.56,0.00
1,TF-IDF + Logistic Regression,7.87,23.15,1.06,23.03,23.43,23.43,58.36,23.43
2,Zero-shot,41.48,54.89,39.65,58.93,45.65,45.65,68.59,45.65
3,XLM-R seed 42,41.45,51.26,44.40,58.51,34.71,34.71,63.37,34.71
4,XLM-R seed 123,40.67,51.03,42.18,57.08,37.21,37.21,63.08,37.21
5,XLM-R seed 1024,39.56,48.35,35.39,48.02,49.10,49.10,59.75,49.10
6,mDeBERTa seed 42,30.44,30.44,0.00,0.00,100.00,100.00,30.44,100.00
7,mDeBERTa seed 123,30.44,30.44,0.00,0.00,100.00,100.00,30.44,100.00
8,mDeBERTa seed 1024,15.00,18.11,0.04,4.51,49.18,49.18,48.95,48.15
9,PhoBERT seed 1024,43.55,56.14,48.77,66.88,31.61,31.61,67.51,31.61


Kết quả fine-tune qua 3 seed: trung bình +- độ lệch chuẩn (ddof=0).


,Kiến trúc,Overall EM (%),Overall F1 (%),HasAns EM (%),HasAns F1 (%),NoAns EM (%),NoAns F1 (%),Accuracy có/không đáp án (%),Recall NoAns (%)
0,XLM-R,40.56 +- 0.77,50.21 +- 1.32,40.66 +- 3.83,54.54 +- 4.64,40.34 +- 6.28,40.34 +- 6.28,62.07 +- 1.64,40.34 +- 6.28
1,mDeBERTa,25.29 +- 7.28,26.33 +- 5.81,0.01 +- 0.02,1.50 +- 2.12,83.06 +- 23.96,83.06 +- 23.96,36.61 +- 8.73,82.72 +- 24.44
2,PhoBERT,44.08 +- 0.49,56.01 +- 0.09,48.27 +- 0.45,65.43 +- 1.04,34.51 +- 2.09,34.51 +- 2.09,66.84 +- 0.62,34.51 +- 2.09


Đã lưu bảng so sánh dev: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\dev_model_comparison.json


## Bước 6 — Phân loại và đối chiếu lỗi trên dev (T18)

Ba nhóm không chồng lặp:
1. Câu có đáp án nhưng dự đoán là chuỗi rỗng.
2. Câu không có đáp án nhưng dự đoán là chuỗi không rỗng.
3. Câu có đáp án và có dự đoán, nhưng không khớp hoàn toàn đáp án chuẩn sau chuẩn hóa (`EM = 0`). Nhóm này giúp tìm các ca chọn sai đoạn hoặc sai biên để đọc lại theo context. EM tự nó không phân biệt hai dạng đó và cũng không khẳng định câu trả lời sai về ngữ nghĩa.

Tỷ lệ trong bảng tính trên toàn bộ 3.814 câu dev của từng seed. Bảng ví dụ so sánh cùng ID giữa ba kiến trúc, mỗi kiến trúc dùng seed F1 cao nhất; nếu bằng điểm thì lấy seed nhỏ hơn. File JSON giữ cả context, các đáp án chuẩn, chuỗi gốc và điểm của từng kiến trúc. Ký hiệu ⟨rỗng⟩ chỉ dùng khi hiển thị.


In [19]:
import json

import pandas as pd

from src.evaluation import score_prediction
from src.utils import save_json

# Bước 5 đã chốt các mã lượt. Đối chiếu vì Bước 6 dùng dự đoán nạp ở Bước 2.
comparison_path = project_root / "results" / "dev_model_comparison.json"
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
if (comparison["split"] != "dev" or
        comparison["finetune_experiment_ids"] != versioned_runs or
        comparison["sample_count"] != len(dev_by_id)):
    raise ValueError("Bảng Bước 5 chưa khớp dữ liệu dev và mã lượt đã nạp ở Bước 2.")

error_labels = {
    "missed_answer": "Có đáp án nhưng trả rỗng",
    "spurious_answer": "Không có đáp án nhưng vẫn trả lời",
    "span_mismatch": "Có đáp án, có trả lời nhưng EM = 0",
}


def classify_error(is_impossible, prediction_text, em):
    """Ví dụ có đáp án Hà Nội, dự đoán '' -> missed_answer."""
    if is_impossible:
        return "spurious_answer" if prediction_text != "" else None
    if prediction_text == "":
        return "missed_answer"
    if em == 0.0:
        return "span_mismatch"
    return None


architectures = ("XLM-R", "mDeBERTa", "PhoBERT")
error_counts = []
chosen_methods = {}

# Chọn seed F1 cao nhất theo bảng chung để minh họa. Nếu bằng điểm, lấy seed nhỏ hơn.
for architecture in architectures:
    best_seed = max(
        (42, 123, 1024),
        key=lambda seed: (
            comparison["methods"][f"{architecture} seed {seed}"]["overall_f1_percent"],
            -seed,
        ),
    )
    chosen_methods[architecture] = f"{architecture} seed {best_seed}"

# Chỉ giữ kết quả từng mẫu của ba seed đại diện để tạo bảng ví dụ.
representative_rows = {}
for architecture in architectures:
    for seed in (42, 123, 1024):
        name = f"{architecture} seed {seed}"
        by_id = {row["id"]: row["prediction_text"] for row in predictions_by_method[name]}
        if set(by_id) != set(dev_by_id):
            raise ValueError(f"{name}: ID dự đoán không khớp dev.")
        counts = {key: 0 for key in error_labels}
        save_examples = name == chosen_methods[architecture]
        if save_examples:
            representative_rows[architecture] = {}

        for question_id, gold in dev_by_id.items():
            prediction_text = by_id[question_id]
            references = gold["answers"]["text"]
            em, f1 = score_prediction(prediction_text, references)
            error_type = classify_error(gold["is_impossible"], prediction_text, em)
            if error_type is not None:
                counts[error_type] += 1

            if save_examples:
                representative_rows[architecture][question_id] = {
                    "seed": seed,
                    "prediction_text": prediction_text,
                    "em": em,
                    "f1": f1,
                    "error_type": error_type,
                }

        # Hai nhóm lỗi về quyết định trả rỗng phải khớp ma trận Bước 5.
        matrix = comparison["methods"][name]["confusion_matrix"]
        if counts["missed_answer"] != matrix[0][1] or counts["spurious_answer"] != matrix[1][0]:
            raise ValueError(f"{name}: số lỗi chuỗi rỗng không khớp ma trận nhầm lẫn.")

        row = {"architecture": architecture, "seed": seed, "dev_count": len(dev_by_id)}
        for error_type, count in counts.items():
            row[error_type] = count
            row[f"{error_type}_pct_dev"] = 100 * count / len(dev_by_id)
        error_counts.append(row)

# Chọn tối đa 2 ví dụ mỗi nhóm. Ưu tiên mẫu có ít nhất một kiến trúc xử lý
# đúng để có đối chiếu trên cùng câu hỏi, sau đó ưu tiên context ngắn.
examples_by_type = {}
example_table = []

for error_type, label in error_labels.items():
    candidates = []
    for question_id, gold in dev_by_id.items():
        predictions = {}
        for architecture in architectures:
            predictions[architecture] = representative_rows[architecture][question_id]
        if not any(row["error_type"] == error_type for row in predictions.values()):
            continue
        correct_count = sum(row["error_type"] is None for row in predictions.values())
        candidates.append((-correct_count, len(gold["context"]), question_id))

    examples = []
    used_contexts = set()
    for _, _, question_id in sorted(candidates):
        gold = dev_by_id[question_id]
        if gold["context"] in used_contexts:
            continue
        used_contexts.add(gold["context"])

        details = {
            "id": question_id,
            "question": gold["question"],
            "context": gold["context"],
            "is_impossible": gold["is_impossible"],
            "references": gold["answers"]["text"],
            "predictions": {},
        }
        display_row = {
            "Nhóm lỗi": label, "ID": question_id, "Câu hỏi": gold["question"],
            "Đáp án chuẩn": " | ".join(details["references"]) or "⟨không có đáp án⟩",
        }
        for architecture in architectures:
            result = representative_rows[architecture][question_id]
            details["predictions"][architecture] = result
            column = f"{architecture} (seed {result['seed']})"
            display_row[column] = result["prediction_text"] or "⟨rỗng⟩"
        examples.append(details)
        example_table.append(display_row)
        if len(examples) == 2:
            break
    if not examples:
        raise ValueError(f"Không tìm thấy ví dụ cho nhóm: {label}")
    examples_by_type[error_type] = examples

counts_table = pd.DataFrame(error_counts)
columns = ["architecture", "seed"]
column_labels = {"architecture": "Kiến trúc", "seed": "Seed"}
for error_type, label in error_labels.items():
    columns.extend([error_type, f"{error_type}_pct_dev"])
    column_labels[error_type] = label
    column_labels[f"{error_type}_pct_dev"] = f"{label} (% dev)"
display(counts_table[columns].rename(columns=column_labels).round(2))
display(pd.DataFrame(example_table))

error_report = {
    "split": "dev",
    "sample_count": len(dev_by_id),
    "finetune_experiment_ids": versioned_runs,
    "percentage_denominator": "toàn bộ số câu dev của từng seed",
    "error_labels": error_labels,
    "counts_by_seed": error_counts,
    "representative_seeds": chosen_methods,
    "examples": examples_by_type,
}
error_path = project_root / "results" / "dev_error_analysis.json"
save_json(error_path, error_report)
print(f"Đã lưu phân tích lỗi dev: {len(error_counts)} dòng seed, {len(examples_by_type)} nhóm lỗi.")
print(error_path)


,Kiến trúc,Seed,Có đáp án nhưng trả rỗng,Có đáp án nhưng trả rỗng (% dev),Không có đáp án nhưng vẫn trả lời,Không có đáp án nhưng vẫn trả lời (% dev),"Có đáp án, có trả lời nhưng EM = 0","Có đáp án, có trả lời nhưng EM = 0 (% dev)"
0,XLM-R,42,639,16.75,758,19.87,836,21.92
1,XLM-R,123,679,17.80,729,19.11,855,22.42
2,XLM-R,1024,944,24.75,591,15.50,770,20.19
3,mDeBERTa,42,2653,69.56,0,0.00,0,0.00
4,mDeBERTa,123,2653,69.56,0,0.00,0,0.00
5,mDeBERTa,1024,1345,35.26,602,15.78,1307,34.27
6,PhoBERT,42,558,14.63,738,19.35,812,21.29
7,PhoBERT,123,510,13.37,749,19.64,878,23.02
8,PhoBERT,1024,445,11.67,794,20.82,914,23.96


,Nhóm lỗi,ID,Câu hỏi,Đáp án chuẩn,XLM-R (seed 42),mDeBERTa (seed 42),PhoBERT (seed 1024)
0,Có đáp án nhưng trả rỗng,0017-0029-0002,Tỉ lệ gia tăng việc làm trung bình hằng năm dư...,"5,3%","5,3%",⟨rỗng⟩,"5,3 %"
1,Có đáp án nhưng trả rỗng,0019-0016-0003,Trong trận nào thì quân Texas đã mất lợi thế h...,trận Alamo,trận Alamo,⟨rỗng⟩,trận Alamo
2,Không có đáp án nhưng vẫn trả lời,0017-0029-0008,Số người có việc làm trong nhiệm kỳ của Roosev...,⟨không có đáp án⟩,⟨rỗng⟩,⟨rỗng⟩,"18,31 triệu"
3,Không có đáp án nhưng vẫn trả lời,0015-0040-0001,Nếu được xếp hạng 1 trong bảng xếp hạng môi tr...,⟨không có đáp án⟩,⟨rỗng⟩,⟨rỗng⟩,tự do nhất
4,"Có đáp án, có trả lời nhưng EM = 0",0019-0016-0001,Ai là tổng thống của México?,Antonio Lopez de Santa Anna,Antonio Lopez de Santa Anna,⟨rỗng⟩,Tổng thống México Antonio Lopez de Santa Anna
5,"Có đáp án, có trả lời nhưng EM = 0",0002-0013-0001,27% dân số là tỷ lệ dân số là người Kanak tại ...,tỉnh Nam,"tỉnh Quần đảo Loyauté, 74% dân số tỉnh Bắc",⟨rỗng⟩,tỉnh Nam


Đã lưu phân tích lỗi dev: 9 dòng seed, 3 nhóm lỗi.
c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\dev_error_analysis.json


## Bước 7 — Giải thích kết quả và độ ổn định (T19)

Số liệu dưới đây lấy từ [bảng chấm chung trên dev](../results/dev_model_comparison.json) và [log mDeBERTa](../results/finetune/runs/run-a28388a1c7754133/mdeberta/diagnostics-seeds-42-123-1024.json). Mỗi seed dùng 3.814 câu dev, gồm 2.653 câu có đáp án và 1.161 câu không có đáp án. Dạng `mean +- std` dùng độ lệch chuẩn quần thể (`ddof=0`) qua ba seed 42, 123, 1024.

| Kiến trúc | Overall EM (%) | Overall F1 (%) | F1 HasAns (%) | Recall NoAns (%) |
|---|---:|---:|---:|---:|
| XLM-R | 40,56 +- 0,77 | 50,21 +- 1,32 | 54,54 | 40,34 |
| mDeBERTa | 25,29 +- 7,28 | 26,33 +- 5,81 | 1,50 | 82,72 |
| PhoBERT | 44,08 +- 0,49 | 56,01 +- 0,09 | 65,43 | 34,51 |

**Quan sát qua ba kiến trúc.** PhoBERT có F1 tổng thể cao nhất trong ba kiến trúc và F1 ít dao động nhất trong ba seed đã chạy. F1 HasAns đạt 65,43%, trong khi recall NoAns là 34,51%, thấp hơn XLM-R (40,34%). XLM-R đứng giữa về F1 tổng thể. mDeBERTa có F1 thấp nhất và dao động lớn nhất (std 5,81 điểm phần trăm). Recall NoAns cao của mDeBERTa đi kèm F1 HasAns rất thấp, nên cần đọc hai nhóm cùng nhau. Các so sánh này giới hạn trên dev với các cấu hình thực nghiệm đã dùng.

| Seed mDeBERTa | Trả rỗng / 3.814 | HasAns EM / F1 (%) | Recall NoAns (%) | Epoch được chọn |
|---|---:|---:|---:|---:|
| 42 | 3.814 (100,00%) | 0,00 / 0,00 | 100,00 | 3 |
| 123 | 3.814 (100,00%) | 0,00 / 0,00 | 100,00 | 2 |
| 1024 | 1.904 (49,92%) | 0,04 / 4,51 | 48,15 | 1 |

Seed 42 và 123 đều bỏ sót cả 2.653 câu HasAns. Vì 1.161/3.814 câu là NoAns, chiến lược luôn trả rỗng vẫn đạt F1 tổng thể 30,44%. Seed 1024 bỏ sót 1.345/2.653 câu HasAns, trả rỗng đúng 559/1.161 câu NoAns và chỉ khớp hoàn toàn 1 câu HasAns. Trong nhóm NoAns của seed 1024, 571 câu có EM = 1 nhưng chỉ 559 dự đoán rỗng nguyên bản: 12 dự đoán còn lại trở thành rỗng sau chuẩn hóa của scorer. Đây là lý do phải tách NoAns EM/F1 khỏi recall tính bằng chuỗi rỗng nguyên bản.

**Đối chiếu log từng epoch.** Các số bên dưới khớp với `trainer_state.json` ở checkpoint cuối của từng seed. `NaN` là giá trị không hữu hạn ghi trong validation loss.

| Seed | F1 dev ở epoch 1 → 2 → 3 (%) | Validation loss ở epoch 1 → 2 → 3 |
|---|---|---|
| 42 | 17,60 → 17,60 → 30,44 | 5,9375 → 5,9375 → NaN |
| 123 | 17,77 → 30,44 → 30,44 | 5,9450 → NaN → NaN |
| 1024 | 18,11 → 17,60 → 17,60 | 5,9472 → 5,9375 → 5,9375 |

Ở seed 42 và 123, validation loss không hữu hạn xuất hiện ở các epoch đạt mức F1 của chiến lược luôn trả rỗng. Ở seed 1024, validation loss vẫn hữu hạn và giảm nhẹ, trong khi F1 giảm sau epoch 1. Điều này giải thích vì sao chọn checkpoint theo F1 dev giữ epoch 1 của seed 1024.

**Giả thuyết cần kiểm tra.** Log cho thấy dấu hiệu bất ổn số ở seed 42/123, nhưng chưa xác định nguồn là trọng số, logits hay phép tính loss. Cả ba seed dùng BF16 nên quan sát này chưa đủ để quy nguyên nhân cho BF16. Nếu tiếp tục thí nghiệm, kiểm tra giá trị hữu hạn của logits start/end và loss trên cùng vài mẫu ở checkpoint trước và sau epoch có NaN. Nếu logits hữu hạn, kiểm tra chênh lệch điểm CLS và span tốt nhất quanh ngưỡng 0, cùng trường hợp decoder không tìm được span hợp lệ. Chỉ kết luận về độ chính xác số hoặc ngưỡng sau khi có kết quả đối chứng.
